##### Why we use `ast.literal_eval`?

- Databricks **widgets** return only **strings**, but you may need the **value** as:
  - `list`
  - `tuple`
  - `dict`
  - `number`
- **literal_eval** safely **transforms** that **string** into the actual Python **datatype**.

**It accepts:**
- **single quotes ' '**
- Python-style **lists, tuples, dicts**
- Python `booleans` **True/False**
- But **does NOT** accept **strict JSON** format only.
- **trailing commas**

| Widget String        | Parsed As                        |
| -------------------- | -------------------------------- |
| `"['id','name']"`    | `['id','name']`                  |
| `"['A101', 'A102']"` | `['A101', 'A102']` (Python list) |
| `"{'x': 1, 'y': 2}"` | `{'x': 1, 'y': 2}`               |
| `"('a','b')"`        | `('a','b')`                      |
| `True`               | `True`                           |
| `False`              | `False`                          |

| Feature                          | `json.loads()` | `ast.literal_eval()`       |
| -------------------------------- | -------------- | -------------------------- |
| Accepts single quotes            | ❌ No           | ✔ Yes                      |
| Accepts trailing commas          | ❌ No           | ✔ Yes                      |
| Accepts Python booleans (`True`) | ❌ No           | ✔ Yes                      |
| Accepts JSON only                | ✔ Yes          | ❌ No (Python objects only) |
| Safe to evaluate                 | ✔ Yes          | ✔ Yes                      |


##### When should you use which?
- **literal_eval** is more **flexible** and **parses Python syntax**.
- **json.loads** is **strict** and **parses only JSON syntax**.

|                            Use Case                                       | Recommended Method |
| ------------------------------------------------------------------------- | ------------------ |
| Widget value is Python-like (**single quotes, tuples, dicts**)            | `ast.literal_eval` |
| Widget value is true **JSON** (**double quotes, strictly formatted**)     | `json.loads`       |
| You want strict validation                                                | `json.loads`       |
| You want flexibility                                                      | `ast.literal_eval` |

      ['A1', 'A2']

- **ast.literal_eval** → ✔ Works
- **json.loads** → ❌ Error
  - Because **JSON does not** allow **single quotes**.

     ["A1","A2"]

- **json.loads** → ✔ Works
- **ast.literal_eval** → ✔ Works
- **Both** will load **correctly**.

##### 1) Accepts single quotes
- `json.loads() → Fails`
- `ast.literal_eval() → Works`

In [0]:
import ast
import json

In [0]:
%skip
dbutils.widgets.removeAll()

In [0]:
# Error: JSON requires double quotes
source_col = "{'name': 'John', 'age': 25}"
json.loads(source_col)

---------------------------------------------------------------------------
JSONDecodeError                           Traceback (most recent call last)
File <command-8073863742116479>, line 3
      1 # Error: JSON requires double quotes
      2 source_col = "{'name': 'John', 'age': 25}"
----> 3 json.loads(source_col)

File /usr/lib/python3.12/json/__init__.py:346, in loads(s, cls, object_hook, parse_float, parse_int, parse_constant, object_pairs_hook, **kw)
    341     s = s.decode(detect_encoding(s), 'surrogatepass')
    343 if (cls is None and object_hook is None and
    344         parse_int is None and parse_float is None and
    345         parse_constant is None and object_pairs_hook is None and not kw):
--> 346     return _default_decoder.decode(s)
    347 if cls is None:
    348     cls = JSONDecoder

File /usr/lib/python3.12/json/decoder.py:337, in JSONDecoder.decode(self, s, _w)
    332 def decode(self, s, _w=WHITESPACE.match):
    333     """Return the Python representation 

In [0]:
# ["sales_id", "product_name", "order_name", "order_id", "insurer_name"]
# ['sales_id', 'product_name', 'order_name', 'order_id', 'insurer_name']

dbutils.widgets.text("source_col", "", "source_col")
source_col = json.loads(dbutils.widgets.get("source_col"))

print(source_col)
print(type(source_col))

---------------------------------------------------------------------------
JSONDecodeError                           Traceback (most recent call last)
File <command-8073863742116480>, line 5
      1 # ["sales_id", "product_name", "order_name", "order_id", "insurer_name"]
      2 # ['sales_id', 'product_name', 'order_name', 'order_id', 'insurer_name']
      4 dbutils.widgets.text("source_col", "", "source_col")
----> 5 source_col = json.loads(dbutils.widgets.get("source_col"))
      7 print(source_col)
      8 print(type(source_col))

File /usr/lib/python3.12/json/__init__.py:346, in loads(s, cls, object_hook, parse_float, parse_int, parse_constant, object_pairs_hook, **kw)
    341     s = s.decode(detect_encoding(s), 'surrogatepass')
    343 if (cls is None and object_hook is None and
    344         parse_int is None and parse_float is None and
    345         parse_constant is None and object_pairs_hook is None and not kw):
--> 346     return _default_decoder.decode(s)
    347 if cl

In [0]:
source_col = "{'name': 'John', 'age': 25}"
result = ast.literal_eval(source_col)

print(result)
print(type(source_col))
print(type(result))

{'name': 'John', 'age': 25}
<class 'str'>
<class 'dict'>


In [0]:
dbutils.widgets.text("source_col", "", "source_col")
source_col = ast.literal_eval(dbutils.widgets.get("source_col"))

print(source_col)
print(type(source_col))

##### 2) Accepts trailing commas
- `json.loads() → Fails`
- `ast.literal_eval() → Works`

In [0]:
# Error: trailing comma not allowed in JSON
source_col = '{"name": "John", "age": 25,}'
json.loads(source_col)

---------------------------------------------------------------------------
JSONDecodeError                           Traceback (most recent call last)
File <command-6082882936634502>, line 3
      1 # Error: trailing comma not allowed in JSON
      2 source_col = '{"name": "John", "age": 25,}'
----> 3 json.loads(source_col)

File /usr/lib/python3.12/json/__init__.py:346, in loads(s, cls, object_hook, parse_float, parse_int, parse_constant, object_pairs_hook, **kw)
    341     s = s.decode(detect_encoding(s), 'surrogatepass')
    343 if (cls is None and object_hook is None and
    344         parse_int is None and parse_float is None and
    345         parse_constant is None and object_pairs_hook is None and not kw):
--> 346     return _default_decoder.decode(s)
    347 if cls is None:
    348     cls = JSONDecoder

File /usr/lib/python3.12/json/decoder.py:337, in JSONDecoder.decode(self, s, _w)
    332 def decode(self, s, _w=WHITESPACE.match):
    333     """Return the Python represe

In [0]:
dbutils.widgets.text("source_col", "", "source_col")
source_col = json.loads(dbutils.widgets.get("source_col"))

print(source_col)
print(type(source_col))

In [0]:
source_col = "{'name': 'John', 'age': 25,}"
result = ast.literal_eval(source_col)

print(result)
print(type(result))

{'name': 'John', 'age': 25}
<class 'dict'>


In [0]:
dbutils.widgets.text("source_col", "", "source_col")
source_col = ast.literal_eval(dbutils.widgets.get("source_col"))

print(source_col)
print(type(source_col))

##### 3) Accepts Python booleans (True, False)
- `json.loads() → Fails`
- `ast.literal_eval() → Works`

In [0]:
# Error → JSON expects true (lowercase)
source_col = '{"status": True}'
json.loads(source_col)

---------------------------------------------------------------------------
JSONDecodeError                           Traceback (most recent call last)
File <command-5098411415282111>, line 3
      1 # Error → JSON expects true (lowercase)
      2 source_col = '{"status": True}'
----> 3 json.loads(source_col)

File /usr/lib/python3.12/json/__init__.py:346, in loads(s, cls, object_hook, parse_float, parse_int, parse_constant, object_pairs_hook, **kw)
    341     s = s.decode(detect_encoding(s), 'surrogatepass')
    343 if (cls is None and object_hook is None and
    344         parse_int is None and parse_float is None and
    345         parse_constant is None and object_pairs_hook is None and not kw):
--> 346     return _default_decoder.decode(s)
    347 if cls is None:
    348     cls = JSONDecoder

File /usr/lib/python3.12/json/decoder.py:337, in JSONDecoder.decode(self, s, _w)
    332 def decode(self, s, _w=WHITESPACE.match):
    333     """Return the Python representation of ``s``

In [0]:
dbutils.widgets.text("source_col", "", "source_col")
source_col = json.loads(dbutils.widgets.get("source_col"))

print(source_col)
print(type(source_col))

In [0]:
source_col = "{'status': True}"
result = ast.literal_eval(source_col)

print(result)
print(type(result))

{'status': True}
<class 'dict'>


In [0]:
dbutils.widgets.text("source_col", "", "source_col")
source_col = ast.literal_eval(dbutils.widgets.get("source_col"))

print(source_col)
print(type(source_col))

##### 4) Accepts JSON only
- `json.loads() → Strict JSON`
- `ast.literal_eval() → Not limited to JSON`

In [0]:
data = '{"name": "Alice", "age": 30}'
print(json.loads(data))

{'name': 'Alice', 'age': 30}


In [0]:
dbutils.widgets.text("source_col", "", "source_col")
source_col = json.loads(dbutils.widgets.get("source_col"))

print(source_col)
print(type(source_col))

In [0]:
# JSON cannot handle tuples, but literal_eval can.
data = "(1, 2, 3)"   # tuple (not JSON!)
print(ast.literal_eval(data))
print(type(data))
print(type(ast.literal_eval(data)))

(1, 2, 3)
<class 'str'>
<class 'tuple'>


In [0]:
dbutils.widgets.text("source_col", "", "source_col")
source_col = ast.literal_eval(dbutils.widgets.get("source_col"))

print(source_col)
print(type(source_col))

(1, 2, 3)
<class 'tuple'>


##### 5) Use Case 01

In [0]:
%skip
# columns to check for upsert
upsert_col = ["SALES_ID", "TARGET_ID"]

# input cols
input_col = ["sales_id", "product_name", "order_name", "order_id", "insurer_name"]
          = {'Age': 27, 'Exp': 5, 'Salary': 100000, 'City': 'Bangalore', 'State': 'KA', 'Country': 'India', 'Gender': 'Female', 'Hobby': 'Dance', 'Marital_Status':'Married'}

          {"Age": 27, "Exp": 5, "Salary": 100000, "City": "Bangalore", "State": "KA", "Country": "India", "Gender": "Female", "Hobby": "Dance", "Marital_Status":"Married"}

target_codes = ["TN~", "AP~", "KA~", "TPT~"]

In [0]:
dbutils.widgets.text("input_col", "", "input_col")
dbutils.widgets.text("upsert_col", "", "upsert_col")
dbutils.widgets.text("target_codes", "", "target_codes")

upsert_col = ast.literal_eval(dbutils.widgets.get("upsert_col"))
input_col = ast.literal_eval(dbutils.widgets.get("input_col"))
target_codes = ast.literal_eval(dbutils.widgets.get("target_codes"))

print(upsert_col)
print(type(upsert_col))

print("\n", input_col)
print(type(input_col))

print("\n", target_codes)
print(type(target_codes))

['SALES_ID', 'TARGET_ID']
<class 'list'>

 {'Age': 27, 'Exp': 5, 'Salary': 100000, 'City': 'Bangalore', 'State': 'KA', 'Country': 'India', 'Gender': 'Female', 'Hobby': 'Dance', 'Marital_Status': 'Married'}
<class 'dict'>

 ['TN~', 'AP~', 'KA~', 'TPT~']
<class 'list'>


In [0]:
upsert_col = json.loads(dbutils.widgets.get("upsert_col"))
input_col = json.loads(dbutils.widgets.get("input_col"))
target_codes = json.loads(dbutils.widgets.get("target_codes"))

print(upsert_col)
print(type(upsert_col))

print("\n", input_col)
print(type(input_col))

print("\n", target_codes)
print(type(target_codes))

['SALES_ID', 'TARGET_ID']
<class 'list'>

 {'Age': 27, 'Exp': 5, 'Salary': 100000, 'City': 'Bangalore', 'State': 'KA', 'Country': 'India', 'Gender': 'Female', 'Hobby': 'Dance', 'Marital_Status': 'Married'}
<class 'dict'>

 ['TN~', 'AP~', 'KA~', 'TPT~']
<class 'list'>


##### 6) Use Case 02

In [0]:
dbutils.widgets.text("list_ordered_clms", "", "list_ordered_clms")
widget_value = dbutils.widgets.get("list_ordered_clms")
print(widget_value)
print(type(widget_value))

\n
<class 'str'>


In [0]:
import ast

dbutils.widgets.text("list_ordered_clms", "", "list_ordered_clms")
widget_value = dbutils.widgets.get("list_ordered_clms")
print(widget_value)
print(type(widget_value))

try:
    if widget_value:
        list_ordered_clms_wo_strip = ast.literal_eval(widget_value)
        print("\nif: ", list_ordered_clms_wo_strip)
        print(type(list_ordered_clms_wo_strip))
    else:
        list_ordered_clms_wo_strip = []
        print("\nelse: ", list_ordered_clms_wo_strip)
        print(type(list_ordered_clms_wo_strip))
except Exception:
    list_ordered_clms_wo_strip = []
    print("\nException: ", list_ordered_clms_wo_strip)
    print(type(list_ordered_clms_wo_strip))

\n
<class 'str'>

Exception:  []
<class 'list'>


     # ast.literal_eval() ignores whitespace
     ast.literal_eval("   ['col1', 'col2']   ")

In [0]:
import ast

dbutils.widgets.text("list_ordered_clms", "", "list_ordered_clms")
widget_value = dbutils.widgets.get("list_ordered_clms")
print(widget_value)
print(type(widget_value))

try:
    if widget_value.strip():
        list_ordered_clms_wo_strip = ast.literal_eval(widget_value)
        print("\nif: ", list_ordered_clms_wo_strip)
        print(type(list_ordered_clms_wo_strip))
    else:
        list_ordered_clms_wo_strip = []
        print("\nelse: ", list_ordered_clms_wo_strip)
        print(type(list_ordered_clms_wo_strip))
except Exception:
    list_ordered_clms_wo_strip = []
    print("\nException: ", list_ordered_clms_wo_strip)
    print(type(list_ordered_clms_wo_strip))

\n
<class 'str'>

Exception:  []
<class 'list'>


**Widget inputs that go to if condition:**

`if widget_value.strip():`
- `The input must be non-empty AFTER removing spaces.`

|         input                        | output                                                                   |
|---------------------------------------|-------------------------------------------------------------------------|
| "suresh "                             | after strip() is "suresh", which is not empty, so enters IF block       |
| ["col1", "col2"]                      | Valid list                                                              |
|    ["col1", "col2"]                   | With leading/trailing spaces; (strip() removes spaces → still non-empty) |
| []                                    | Empty list; "[]".strip() → "[]" → TRUE                                  |
| ""                                    | empty string                                                            |
| [["id", "name"], ["age", "salary"]]   | Nested list                                                             |
| Numbers list                          | [1, 2, 3]                                                               |
| Tuple                                 | ("a", "b", "c")                                                         |
| Dictionary                            | {"a": 1, "b": 2}                                                        |
| Boolean / number                      | True; 10                                                                |

      # Standard list of columns
      list_ordered_clms = ["Sales_DT", "Product_ID" , "Product_NM", "Product_Group", "Source_NM"   , "Target_NM", "Device_Category", "Company_NM", "Grade", "Event_Name", "Event_Type", "Sessions_CNT"]

      # Single column
      ["id"]

**Widget inputs that go to else condition:**
- `Completely empty`
- `Only spaces`
  - "     "
  - `After strip() → "" → goes to else`.
- `Tabs / spaces mix`
- `Newline characters`

**Widget inputs that go to Exception condition:**
- \n